In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score, recall_score, confusion_matrix
from torch.utils.data import DataLoader, WeightedRandomSampler, TensorDataset
from sklearn.preprocessing import StandardScaler
import joblib
import os
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

Używane urządzenie: cuda


In [3]:
COLUMNS = ["loan_amnt", "term", "installment", "purpose", "annual_inc",  
        "emp_length", "home_ownership", "dti", "delinq_2yrs", "mths_since_last_delinq", "pub_rec",
        "inq_last_6mths", "open_acc", "total_acc", "revol_bal", "revol_util", "acc_now_delinq",
        "mort_acc", "earliest_cr_line", "avg_cur_bal", "num_tl_op_past_12m", "pct_tl_nvr_dlq",
        "chargeoff_within_12_mths", "mo_sin_old_rev_tl_op", "tot_cur_bal",
        "bc_util", "num_actv_bc_tl", "num_tl_90g_dpd_24m", "tot_coll_amt",
        "total_rev_hi_lim", "initial_list_status", "inq_fi", "inq_last_12m",
        "issue_d"
]

COLS_IN_PD = ['annual_inc', 'target', 'num_tl_op_past_12m', 'inq_fi_6m',
 'home_ownership', 'tot_cur_bal', 'total_rev_hi_lim', 'bc_util', 'dti',
 'mo_sin_old_rev_tl_op', 'revol_util', 'loan_amnt'
]

COLS = [col for col in COLUMNS if col not in COLS_IN_PD]
COLS.append("target")

with open("data/columns_grade.json", "w") as f:
    json.dump(COLS, f)

In [4]:
df = pd.read_csv("data/lean_dataset.csv", usecols=COLS)
df['issue_d'] = pd.to_datetime(df['issue_d'])
train_mask = (df['issue_d'] >= '2013-01-01') & (df['issue_d'] <= '2015-12-31')
df = df[train_mask]



C:\Users\User\AppData\Local\Temp\ipykernel_2188\1203992458.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['issue_d'] = pd.to_datetime(df['issue_d'])


In [5]:
df.columns

Index(['term', 'installment', 'emp_length', 'issue_d', 'purpose',
       'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths',
       'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal',
       'total_acc', 'initial_list_status', 'acc_now_delinq', 'tot_coll_amt',
       'inq_fi', 'inq_last_12m', 'avg_cur_bal', 'chargeoff_within_12_mths',
       'mort_acc', 'num_actv_bc_tl', 'num_tl_90g_dpd_24m', 'pct_tl_nvr_dlq',
       'target'],
      dtype='object')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 729275 entries, 311111 to 1213200
Data columns (total 25 columns):
 #   Column                    Non-Null Count   Dtype         
---  ------                    --------------   -----         
 0   term                      729275 non-null  object        
 1   installment               729275 non-null  float64       
 2   emp_length                689893 non-null  object        
 3   issue_d                   729275 non-null  datetime64[ns]
 4   purpose                   729275 non-null  object        
 5   delinq_2yrs               729275 non-null  float64       
 6   earliest_cr_line          729275 non-null  object        
 7   inq_last_6mths            729275 non-null  float64       
 8   mths_since_last_delinq    364155 non-null  float64       
 9   open_acc                  729275 non-null  float64       
 10  pub_rec                   729275 non-null  float64       
 11  revol_bal                 729275 non-null  int64         
 12  t

In [7]:
correlation = df['inq_fi'].corr(df['inq_last_12m'])
identical_rows = (df['inq_fi'] == df['inq_last_12m']).sum()
total_rows = len(df)

print(f"Korelacja: {correlation:.4f}")
print(f"Identyczne wiersze: {identical_rows} z {total_rows} ({identical_rows/total_rows:.2%})") # if NaN in the same row it should be more than 2.56%

Korelacja: 0.5647
Identyczne wiersze: 5506 z 729275 (0.75%)


In [8]:
def prepare_df(df):
    df['term_60m'] = df['term'].str.extract('(\d+)').astype(float)
    df['term_60m'] = (df['term_60m'] == 60).astype(int)
    df = df.drop(columns=['term'])


    df['emp_length_is_null'] = df['emp_length'].isnull().astype(int)
    df['emp_length_num'] = df['emp_length'].str.extract('(\d+)').astype(float)
    df['emp_length_num'] = df['emp_length_num'].fillna(-1)


    df['inq_fi'] = df['inq_fi'].fillna(-1)
    df['inq_last_12m'] = df['inq_last_12m'].fillna(-1)
    df['fi_to_total_ratio'] = np.where(
        df['inq_last_12m'] > 0, 
        df['inq_fi'] / df['inq_last_12m'], 
        0
    ) 


    df['is_never_delinq'] = df['mths_since_last_delinq'].isnull().astype(int)
    df['mths_since_last_delinq'] = df['mths_since_last_delinq'].fillna(-1)


    df['issue_d'] = pd.to_datetime(df['issue_d'])
    df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])
    df['credit_hist_months'] = (
        (df['issue_d'].dt.year - df['earliest_cr_line'].dt.year) * 12 +
        (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month)
    )


    df['credit_hist_months'] = df['credit_hist_months'].apply(lambda x: max(x, 0))
    df['credit_hist_months'] = df['credit_hist_months'].fillna(-1) # There aren't any NaN values in this dataset, just to be sure


    cols_to_transform = ["revol_bal", "avg_cur_bal"]
    for col in cols_to_transform:
        upper_limit = df[col].quantile(0.99)
        df[col] = df[col].clip(upper=upper_limit)
        df[col] = np.log10(df[col] + 1)


    cat_cols = ['purpose']
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


    to_drop = ['application_type', 'emp_title', 'issue_d', 'loan_status', 'initial_list_status', # Initial status as I counldn't find whether you can do fractional loans in Poland
            'earliest_cr_line', 'emp_length', 'earliest_cr_line'] 
    df = df.drop(columns=[c for c in to_drop if c in df.columns])


    df['inq_recent_ratio'] = df['inq_last_6mths'] / (df['inq_last_12m'] + 0.001)
    df['utilization_proxy'] = df['revol_bal'] / (df['avg_cur_bal'] + 0.001)

    df = df.fillna(0)
    return df

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
C:\Users\User\AppData\Local\Temp\ipykernel_2188\2641278464.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['term_60m'] = df['term'].str.extract('(\d+)').astype(float)
C:\Users\User\AppData\Local\Temp\ipykernel_2188\2641278464.py:8: SyntaxWarning: invalid escape sequence '\d'
  df['emp_length_num'] = df['emp_length'].str.extract('(\d+)').astype(float)


# NN

In [9]:
df = prepare_df(df)

C:\Users\User\AppData\Local\Temp\ipykernel_2188\2641278464.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])


In [10]:
df_test = pd.read_csv("data/lean_dataset.csv", usecols=COLS)
df_test['issue_d'] = pd.to_datetime(df_test['issue_d'])
test_mask  = (df_test['issue_d'] >= '2016-01-01') & (df_test['issue_d'] <= '2016-12-31')
df_test = df_test[train_mask]

df_test = prepare_df(df_test)

C:\Users\User\AppData\Local\Temp\ipykernel_2188\678585288.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_test['issue_d'] = pd.to_datetime(df_test['issue_d'])
C:\Users\User\AppData\Local\Temp\ipykernel_2188\2641278464.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 729275 entries, 311111 to 1213200
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   installment                 729275 non-null  float64
 1   delinq_2yrs                 729275 non-null  float64
 2   inq_last_6mths              729275 non-null  float64
 3   mths_since_last_delinq      729275 non-null  float64
 4   open_acc                    729275 non-null  float64
 5   pub_rec                     729275 non-null  float64
 6   revol_bal                   729275 non-null  float64
 7   total_acc                   729275 non-null  float64
 8   acc_now_delinq              729275 non-null  float64
 9   tot_coll_amt                729275 non-null  float64
 10  inq_fi                      729275 non-null  float64
 11  inq_last_12m                729275 non-null  float64
 12  avg_cur_bal                 729275 non-null  float64
 13  chargeoff_wit

In [12]:
X = df.drop(columns=['target'])
y = df['target'].values

feature_order = X.columns.tolist() 

scaler = StandardScaler()
scaler.fit(X[feature_order])
joblib.dump({'scaler': scaler, 'feature_order': feature_order}, 'metadata.joblib')


X_scaled = scaler.transform(X[feature_order])

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
dataset = TensorDataset(X_tensor, y_tensor)

class_sample_count = np.bincount(y.astype(int))
weights = 1. / torch.tensor(class_sample_count, dtype=torch.float)
samples_weights = weights[y.astype(int)]

sampler = WeightedRandomSampler(
    weights=samples_weights, 
    num_samples=len(samples_weights), 
    replacement=True
)

train_loader = DataLoader(
    dataset, 
    batch_size=1024, 
    sampler=sampler
)


In [13]:
X_test = df_test.drop(columns=['target'])
y_test = df_test['target'].values

X_test_scaled = scaler.transform(X_test[feature_order])

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(
    test_dataset, 
    batch_size=1024, 
    shuffle=False
)


In [14]:
class FlexibleScoringNet(nn.Module):
    def __init__(self, input_dim, layers_list, dropout_rate=0.2):
        super(FlexibleScoringNet, self).__init__()
        
        layers = []
        in_features = input_dim
        
        for h_size in layers_list:
            layers.append(nn.Linear(in_features, h_size))
            layers.append(nn.BatchNorm1d(h_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_features = h_size
        
        layers.append(nn.Linear(in_features, 1))
        
        self.model = nn.Sequential(*layers).to(device)
        
    def forward(self, x):
        return self.model(x)

input_dim = X_tensor.shape[1]

In [15]:
os.makedirs("models", exist_ok=True)

In [16]:
def evaluate(model, X_tensor, y_tensor, return_cm=False):
    model.eval()
    X_tensor = X_tensor.to(device)
    
    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs > 0.5).astype(int)
        y_true = y_tensor.cpu().numpy()
        
        auc = roc_auc_score(y_true, probs)
        recall_1 = recall_score(y_true, preds)
        
        if return_cm:
            cm = confusion_matrix(y_true, preds)
            return auc, recall_1, cm
            
    return auc, recall_1

architectures = [
    [32],
    [64],
    [128],
    [64, 32], 
    [128, 64, 32],  # Turned out to be "the best"
]
learning_rates = [0.001, 0.0005]

input_dim = X_tensor.shape[1]
PATIENCE = 3
MAX_EPOCHS = 20
final_results = {}

for layers in architectures:
    for lr in learning_rates:
        arch_name = "-".join(map(str, layers))
        test_id = f"Arch_{arch_name}_LR_{lr}"
        
        print(f"\n" + "="*60)
        print(f" TEST: {test_id}")
        print("="*60)
        
        model = FlexibleScoringNet(input_dim, layers).to(device)
        criterion = nn.BCEWithLogitsLoss().to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
        
        best_auc = 0
        epochs_no_improve = 0
        
        for epoch in range(MAX_EPOCHS):
            model.train()
            total_loss = 0
            
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            current_auc, current_recall = evaluate(model, X_test_tensor, y_test_tensor)
            
            scheduler.step(current_auc)
            
            avg_loss = total_loss / len(train_loader)
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoka {epoch+1:02d} | Loss: {avg_loss:.4f} | AUC: {current_auc:.4f} | LR: {current_lr:.6f}")
            
            if current_auc > best_auc:
                best_auc = current_auc
                epochs_no_improve = 0
                torch.save(model.state_dict(), f'models/best_model_{test_id}.pth')
            else:
                epochs_no_improve += 1
                
            if epochs_no_improve >= PATIENCE:
                print(f"Early Stopping. Najlepsze AUC: {best_auc:.4f}")
                break

        model.load_state_dict(torch.load(f'models/best_model_{test_id}.pth'))
        f_auc, f_recall, f_cm = evaluate(model, X_test_tensor, y_test_tensor, return_cm=True)
        
        final_results[test_id] = {'auc': f_auc, 'recall': f_recall, 'cm': f_cm}
        
        print(f"\n\nPODSUMOWANIE DLA {test_id}:")
        print(f"Finalne AUC: {f_auc:.4f} | Finalny Recall: {f_recall:.4f}")
        print(f"Confusion Matrix:\n{f_cm}")

best_id = max(final_results, key=lambda x: final_results[x]['auc'])
print(f"\n\nNajlepsza konfiguracja: {best_id}")
print(f"AUC: {final_results[best_id]['auc']:.4f}")


 TEST: Arch_32_LR_0.001
Epoka 01 | Loss: 0.6491 | AUC: 0.6842 | LR: 0.001000
Epoka 02 | Loss: 0.6418 | AUC: 0.6851 | LR: 0.001000
Epoka 03 | Loss: 0.6403 | AUC: 0.6872 | LR: 0.001000
Epoka 04 | Loss: 0.6397 | AUC: 0.6877 | LR: 0.001000
Epoka 05 | Loss: 0.6397 | AUC: 0.6883 | LR: 0.001000
Epoka 06 | Loss: 0.6389 | AUC: 0.6888 | LR: 0.001000
Epoka 07 | Loss: 0.6380 | AUC: 0.6890 | LR: 0.001000
Epoka 08 | Loss: 0.6390 | AUC: 0.6891 | LR: 0.001000
Epoka 09 | Loss: 0.6396 | AUC: 0.6880 | LR: 0.001000
Epoka 10 | Loss: 0.6386 | AUC: 0.6896 | LR: 0.001000
Epoka 11 | Loss: 0.6379 | AUC: 0.6897 | LR: 0.001000
Epoka 12 | Loss: 0.6381 | AUC: 0.6899 | LR: 0.001000
Epoka 13 | Loss: 0.6384 | AUC: 0.6901 | LR: 0.001000
Epoka 14 | Loss: 0.6374 | AUC: 0.6899 | LR: 0.001000
Epoka 15 | Loss: 0.6379 | AUC: 0.6902 | LR: 0.001000
Epoka 16 | Loss: 0.6387 | AUC: 0.6901 | LR: 0.001000
Epoka 17 | Loss: 0.6382 | AUC: 0.6904 | LR: 0.001000
Epoka 18 | Loss: 0.6374 | AUC: 0.6905 | LR: 0.001000
Epoka 19 | Loss: 0.63

In [17]:
df_test.columns

Index(['installment', 'delinq_2yrs', 'inq_last_6mths',
       'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal',
       'total_acc', 'acc_now_delinq', 'tot_coll_amt', 'inq_fi', 'inq_last_12m',
       'avg_cur_bal', 'chargeoff_within_12_mths', 'mort_acc', 'num_actv_bc_tl',
       'num_tl_90g_dpd_24m', 'pct_tl_nvr_dlq', 'target', 'term_60m',
       'emp_length_is_null', 'emp_length_num', 'fi_to_total_ratio',
       'is_never_delinq', 'credit_hist_months', 'purpose_credit_card',
       'purpose_debt_consolidation', 'purpose_educational',
       'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase',
       'purpose_medical', 'purpose_moving', 'purpose_other',
       'purpose_renewable_energy', 'purpose_small_business',
       'purpose_vacation', 'purpose_wedding', 'inq_recent_ratio',
       'utilization_proxy'],
      dtype='object')

In [19]:
artifacts = joblib.load('metadata.joblib')
scaler = artifacts['scaler']
feature_order = artifacts['feature_order']

In [20]:
artifacts = joblib.load('metadata.joblib')
feature_order = artifacts['feature_order']
scaler = artifacts['scaler']

best_arch_layers = [128, 64, 32]
best_model_path = 'models/best_model_Arch_128-64-32_LR_0.0005.pth'

input_dim = len(feature_order)
model = FlexibleScoringNet(input_dim, best_arch_layers).to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

df_graded = df.copy()

cols_to_log = ["revol_bal", "avg_cur_bal"]
for col in cols_to_log:
    if col in df_graded.columns:
        upper_limit = df_graded[col].quantile(0.99)
        df_graded[col] = df_graded[col].clip(upper=upper_limit)
        df_graded[col] = np.log10(df_graded[col] + 1)

X = df_graded[feature_order]
X_scaled = scaler.transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(device)

with torch.no_grad():
    logits = model(X_tensor)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()

df_raw = pd.read_csv("data/lean_dataset.csv")
df_raw = df_raw[train_mask]
df_raw['probability'] = probs


df_raw.to_csv('data/lean_graded.csv', index=False)